Machine Learning Models

In [ ]:
import os
import json
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import CountVectorizer

# Function to preprocess the data with consistent feature vector length
def preprocess_data_fixed_length(directory):
    all_features = []
    labels = []

    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        with open(file_path, 'r') as json_file:
            data = json.load(json_file)
            # Creating a string representation of features for each graph
            feature_str = ' '.join([f'node{node_id}_feature{feature_value}' for node_id, feature_value in data['features'].items()])
            all_features.append(feature_str)
            labels.append(data['labels'])

    # Using CountVectorizer to create consistent length feature vectors
    vectorizer = CountVectorizer()
    feature_vectors = vectorizer.fit_transform(all_features).toarray()

    return feature_vectors, np.array(labels)

# Directory where the graph2vec_input data is located
graph2vec_input_dir = ''  # Replace with your directory path

# Preprocessing the data with fixed length feature vectors
features_fixed, labels_fixed = preprocess_data_fixed_length(graph2vec_input_dir)

# Splitting the data into training and test sets
X_train_fixed, X_test_fixed, y_train_fixed, y_test_fixed = train_test_split(features_fixed, labels_fixed, test_size=0.3, random_state=42)


# Training the Random Forest Classifier
#change your machine learning model here and modify the code accordingly
clf_fixed = RandomForestClassifier()
clf_fixed.fit(X_train_fixed, y_train_fixed)

# Predicting on thex test set
y_pred_fixed = clf_fixed.predict(X_test_fixed)

# Evaluating the model
classification_report_result_fixed = classification_report(y_test_fixed, y_pred_fixed)
print(classification_report_result_fixed)


CNN

In [ ]:
import os
import json
import numpy as np

def find_max_size(directory):
    max_size = 0
    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        with open(file_path, 'r') as json_file:
            data = json.load(json_file)
            features = data.get('features', {})
            max_node = max(max(map(int, edge)) for edge in data['edges']) if data['edges'] else 0
            max_size = max(max_size, max_node + 1)
    return max_size

graph_data_dir = ''  # Replace with your directory path
max_size = find_max_size(graph_data_dir)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, MaxPooling2D

model = Sequential()
model.add(Conv2D(32, kernel_size=3, activation='relu', input_shape=(max_size, max_size, 1)))
model.add(MaxPooling2D(pool_size=2))
model.add(Conv2D(64, kernel_size=3, activation='relu'))
model.add(MaxPooling2D(pool_size=2))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(y.shape[1], activation='sigmoid'))  # Ensure y.shape[1] is defined or replace with the number of classes

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
model_evaluation = model.evaluate(X_test, y_test)
print("Loss:", model_evaluation[0])
print("Accuracy:", model_evaluation[1])


GNN

In [ ]:
import os
import json
import torch
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

# Function to preprocess the data to graph format
def preprocess_data_graph(directory):
    graphs = []
    labels = []

    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        with open(file_path, 'r') as json_file:
            data = json.load(json_file)
            # Creating edge index tensor
            edge_index = torch.tensor(list(data['edges']), dtype=torch.long)
            # Creating node feature tensor
            node_features = torch.tensor(list(data['features'].values()), dtype=torch.float)
            # Creating graph data object
            graph = Data(x=node_features, edge_index=edge_index.t().contiguous())
            graphs.append(graph)
            labels.append(data['labels'])

    return graphs, torch.tensor(labels, dtype=torch.long)

# Directory where the graph data is located
graph_data_dir = ''  # Replace with your directory path

# Preprocessing the data into graph format
graphs, labels = preprocess_data_graph(graph_data_dir)

# Splitting the data into training and test sets
train_graphs, test_graphs, train_labels, test_labels = train_test_split(graphs, labels, test_size=0.3, random_state=42)

# DataLoader for batch processing
train_loader = DataLoader(train_graphs, batch_size=10, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=10, shuffle=False)

# Define the GCN model
class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

# Assuming node features' dimension and number of classes are known
model = GCN(input_dim=20, hidden_dim=64, output_dim=4)  # Adjust dimensions as necessary
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.NLLLoss()

# Training the model
def train():
    model.train()
    for data in train_loader:
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

# Test the model
def test(loader):
    model.eval()
    correct = 0
    for data in loader:
        out = model(data)
        pred = out.argmax(dim=1)
        correct += pred.eq(data.y).sum().item()
    return correct / len(loader.dataset)

# Run training and testing
train()
accuracy = test(test_loader)
print(f'Accuracy: {accuracy}')


Transformer Approach

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
import pandas as pd

data = pd.read_csv('your_dataset.csv')  # Update the path to your dataset file

# Split the dataset
train_texts, test_texts, train_labels, test_labels = train_test_split(data['text'], data['label'], test_size=0.2, random_state=42)

# Tokenization and Dataset preparation
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
          text,
          add_special_tokens=True,
          max_length=self.max_len,
          return_token_type_ids=False,
          padding='max_length',
          truncation=True,
          return_attention_mask=True,
          return_tensors='pt',
        )
        return {
          'text': text,
          'input_ids': encoding['input_ids'].flatten(),
          'attention_mask': encoding['attention_mask'].flatten(),
          'labels': torch.tensor(label, dtype=torch.long)
        }

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset = TextDataset(train_texts, train_labels, tokenizer)
test_dataset = TextDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

# Model Initialization
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=data['label'].nunique())

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training Function
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0
    for d in data_loader:
        input_ids = d['input_ids'].to(device)
        attention_mask = d['attention_mask'].to(device)
        labels = d['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(data_loader)

# Evaluation Function
def eval_model(model, data_loader, device):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for d in data_loader:
            input_ids = d['input_ids'].to(device)
            attention_mask = d['attention_mask'].to(device)
            labels = d['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs.logits, dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()
    return correct / total

# Run Training and Evaluation
for epoch in range(3):  # Number of epochs
    train_loss = train_epoch(model, train_loader, optimizer, device)
    test_accuracy = eval_model(model, test_loader, device)
    print(f'Epoch {epoch + 1}, Train Loss: {train_loss}, Test Accuracy: {test_accuracy}')
